## imports and setup

In [4]:
import numpy as np
import pandas as pd
import random
import re
from datetime import datetime
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)
random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

OUT_DIR = Path(
    r"F:\UIU\2nd sems course\Data Analytics\Power BI Project\Data Visualization Assignment\Python Files\test")

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Saving files to:")
print(OUT_DIR)

Saving files to:
F:\UIU\2nd sems course\Data Analytics\Power BI Project\Data Visualization Assignment\Python Files\test


## Helper lists and utility functions

In [5]:
# -----------------------------
# Location reference
# -----------------------------
districts = [
    "Dhaka", "Faridpur", "Gazipur", "Gopalganj", "Kishoreganj", "Madaripur",
    "Manikganj", "Munshiganj", "Narayanganj", "Narsingdi", "Rajbari",
    "Shariatpur", "Tangail", "Bandarban", "Brahmanbaria", "Chandpur",
    "Chittagong", "Comilla", "Cox's Bazar", "Feni", "Khagrachari",
    "Lakshmipur", "Noakhali", "Rangamati"
]

district_to_region = {
    "Dhaka": "Dhaka",
    "Faridpur": "Dhaka",
    "Gazipur": "Dhaka",
    "Gopalganj": "Dhaka",
    "Kishoreganj": "Dhaka",
    "Madaripur": "Dhaka",
    "Manikganj": "Dhaka",
    "Munshiganj": "Dhaka",
    "Narayanganj": "Dhaka",
    "Narsingdi": "Dhaka",
    "Rajbari": "Dhaka",
    "Shariatpur": "Dhaka",
    "Tangail": "Dhaka",
    "Bandarban": "Chattogram",
    "Brahmanbaria": "Chattogram",
    "Chandpur": "Chattogram",
    "Chittagong": "Chattogram",
    "Comilla": "Chattogram",
    "Cox's Bazar": "Chattogram",
    "Feni": "Chattogram",
    "Khagrachari": "Chattogram",
    "Lakshmipur": "Chattogram",
    "Noakhali": "Chattogram",
    "Rangamati": "Chattogram",
}

# -----------------------------
# Name pools
# -----------------------------
first_names = [
    "Rahim", "Karim", "Ayesha", "Nusrat", "Sadia", "Tanvir", "Sakib", "Muntasir",
    "Rafi", "Tanjim", "Nayeem", "Jannat", "Imran", "Shaila", "Fahim", "Farhan",
    "Hasan", "Mim", "Raisa", "Nabil", "Tamim", "Adib", "Sami", "Rumana",
    "Shuvo", "Arif", "Nafis", "Tasnima", "Sifat", "Mahin"
]

last_names = [
    "Ahmed", "Islam", "Khan", "Hossain", "Chowdhury", "Rahman", "Sarker",
    "Miah", "Das", "Roy", "Talukder", "Uddin", "Bhuiyan", "Jahan", "Nahar",
    "Amin", "Patwary", "Mollik", "Sheikh", "Karim"
]

genders = ["Male", "Female"]

# -----------------------------
# Book taxonomy
# -----------------------------
book_categories = {
    "Academic": ["Mathematics", "Physics", "Chemistry", "Biology", "English Grammar", "Bangla Literature"],
    "Children": ["Picture Books", "Story Books", "Rhymes", "Activity Books", "Coloring Books"],
    "Fiction": ["Mystery", "Romance", "Thriller", "Adventure", "Classic", "Short Stories"],
    "Business": ["Marketing", "Finance", "Entrepreneurship", "Management", "Accounting"],
    "Programming": ["Python", "Data Science", "Web Development", "Machine Learning", "Algorithms", "Databases"],
    "Religious": ["Quran Studies", "Hadith", "Fiqh", "Islamic History", "Prayer Guides"],
    "Exam Prep": ["BCS", "Bank Jobs", "Admission Test", "University Admission", "Teacher Recruitment"],
    "Self Help": ["Productivity", "Habit Building", "Communication", "Leadership", "Mindfulness"],
    "Biography": ["Leaders", "Scientists", "Writers", "Entrepreneurs", "Artists"]
}

title_styles = ["Essentials", "Guide", "Workbook", "Handbook", "Masterclass", "Simplified", "Complete"]
book_formats = ["Paperback", "Hardcover", "eBook"]

payment_methods = ["Cash", "Card", "bKash", "Nagad", "Rocket"]
sales_channels = ["Online", "In-Store", "App"]

author_first = [
    "Amin", "Jalal", "Rashid", "Mala", "Sultana", "Ferdous", "Anika", "Kamal",
    "Shahana", "Riaz", "Naila", "Touhid", "Sabbir", "Rumana", "Mohiuddin"
]
author_last = ["Rahman", "Ahmed", "Islam", "Hossain", "Khan", "Chowdhury", "Sarker", "Uddin"]

supplier_prefix = ["Dhaka", "Bengal", "Popular", "Ananda", "Bookline", "Prothom", "Nirjhor", "Borno"]
supplier_suffix = ["Publications", "Books", "Media", "Press", "Publishers", "Distributors"]

# -----------------------------
# Utility functions
# -----------------------------
def random_dates(start_date, end_date, n):
    start_ts = pd.Timestamp(start_date).value // 10**9
    end_ts = pd.Timestamp(end_date).value // 10**9
    return pd.to_datetime(rng.integers(start_ts, end_ts, size=n), unit="s")

def make_name(n):
    return [f"{rng.choice(first_names)} {rng.choice(last_names)}" for _ in range(n)]

def make_email(name, domain="gmail.com"):
    slug = re.sub(r"[^a-z0-9]+", ".", str(name).lower()).strip(".")
    return f"{slug}@{domain}"

def make_phone(n):
    # Bangladesh-style mobile format
    return [f"01{rng.integers(3, 10)}{rng.integers(100000000, 999999999):09d}"[:11] for _ in range(n)]

def make_isbn(n):
    # 13-digit synthetic ISBN-like string
    return [f"978{rng.integers(1000000000, 9999999999)}" for _ in range(n)]

def cap_iqr(series):
    s = pd.to_numeric(series, errors="coerce")
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return s.clip(lower, upper)

def introduce_nulls(df, cols, frac=0.03):
    df = df.copy()
    n = max(1, int(len(df) * frac))
    for col in cols:
        idx = rng.choice(df.index, size=n, replace=False)
        df.loc[idx, col] = np.nan
    return df

def introduce_inconsistent_text(df, col, patterns):
    df = df.copy()
    idx = rng.choice(df.index, size=max(1, int(len(df) * 0.10)), replace=False)
    for i in idx:
        val = str(df.loc[i, col])
        df.loc[i, col] = random.choice(patterns)(val)
    return df

## Generate suppliers / publishers

In [6]:
N_SUPPLIERS = 1000

supplier_prefix = [
    "Global", "Eastern", "Prime", "National", "Delta",
    "Green", "Bright", "Ocean", "Royal", "Metro"
]

supplier_suffix = [
    "Books Ltd", "Publishing House", "Distributors",
    "Enterprises", "Logistics", "Supply Co", "Trade Group"
]

supplier_ids = [f"S{str(i).zfill(5)}" for i in range(1, N_SUPPLIERS + 1)]

supplier_df = pd.DataFrame({
    "supplier_id": supplier_ids,

    "supplier_name": [
        f"{rng.choice(supplier_prefix)} {rng.choice(supplier_suffix)}"
        for _ in range(N_SUPPLIERS)
    ],

    "country": rng.choice(
        ["Bangladesh", "India", "UK", "USA", "China"],
        size=N_SUPPLIERS,
        p=[0.72, 0.10, 0.07, 0.06, 0.05]
    ),

    "contact_person": make_name(N_SUPPLIERS),

    "phone": make_phone(N_SUPPLIERS),

    "email": [
        make_email(name, domain=rng.choice(["gmail.com", "yahoo.com", "outlook.com"]))
        for name in make_name(N_SUPPLIERS)
    ],

    "product_category": rng.choice(list(book_categories.keys()), size=N_SUPPLIERS),

    "rating": np.round(rng.uniform(2.5, 5.0, size=N_SUPPLIERS), 2),

    "contract_start_date": random_dates("2019-01-01", "2025-12-31", N_SUPPLIERS),

    "delivery_time_days": rng.integers(2, 20, size=N_SUPPLIERS),

    "city": rng.choice(districts, size=N_SUPPLIERS),

    "status": rng.choice(
        ["Active", "Active", "Active", "Inactive", "Suspended"],
        size=N_SUPPLIERS,
        p=[0.55, 0.20, 0.10, 0.10, 0.05]
    ),
})

supplier_df["region"] = supplier_df["city"].map(district_to_region)

supplier_df.head()

,supplier_id,supplier_name,country,contact_person,phone,email,product_category,rating,contract_start_date,delivery_time_days,city,status,region
0,S00001,Global Supply Co,Bangladesh,Tanjim Patwary,01767617525,nabil.das@gmail.com,Academic,4.61,2020-05-20 11:49:34,8,Gopalganj,Active,Dhaka
1,S00002,Bright Enterprises,Bangladesh,Karim Das,01735268960,hasan.roy@outlook.com,Fiction,4.59,2024-10-19 08:28:38,9,Comilla,Active,Chattogram
2,S00003,Delta Trade Group,Bangladesh,Shuvo Karim,01816283089,tanjim.sarker@gmail.com,Biography,3.57,2023-01-01 18:46:34,6,Noakhali,Active,Chattogram
3,S00004,Global Logistics,Bangladesh,Muntasir Jahan,01345699932,tasnima.jahan@outlook.com,Programming,4.92,2024-12-16 07:17:16,9,Gopalganj,Active,Dhaka
4,S00005,Prime Books Ltd,Bangladesh,Tanvir Khan,01869133177,ayesha.karim@yahoo.com,Biography,4.95,2019-06-09 05:50:23,17,Comilla,Active,Chattogram


## Generate stores / branches

In [7]:
N_STORES = 1000

store_ids = [f"ST{str(i).zfill(5)}" for i in range(1, N_STORES + 1)]

store_brands = [
    "Rokomari", "BookHub", "Pages Cafe",
    "Readers Point", "Literary World",
    "Book Valley", "Story House", "Novel Nest"
]

store_types = ["Flagship", "Standard", "Mini", "Online Hub", "Mall Outlet"]

store_city_weights = np.array(
    [0.30 if d == "Dhaka" else 0.03 for d in districts],
    dtype=float
)

store_city_weights = store_city_weights / store_city_weights.sum()

store_df = pd.DataFrame({
    "store_id": store_ids,

    "store_name": [
        f"{rng.choice(store_brands)} {rng.choice(['Branch', 'Outlet', 'Center', 'Store'])} - {rng.choice(districts)}"
        for _ in range(N_STORES)
    ],

    "city": rng.choice(districts, size=N_STORES, p=store_city_weights),

    "region": None,

    "store_type": rng.choice(
        store_types,
        size=N_STORES,
        p=[0.20, 0.40, 0.20, 0.10, 0.10]
    ),

    "opening_date": random_dates("2017-01-01", "2025-12-31", N_STORES),

    "manager_name": make_name(N_STORES),

    "floor_area": np.round(rng.uniform(400, 6000, size=N_STORES), 2),

    "monthly_rent": np.round(rng.uniform(15000, 250000, size=N_STORES), 2),

    "employee_count": rng.integers(2, 60, size=N_STORES),

    "store_rating": np.round(rng.uniform(2.8, 5.0, size=N_STORES), 2),
})

store_df["region"] = store_df["city"].map(district_to_region)

store_df.head()

,store_id,store_name,city,region,store_type,opening_date,manager_name,floor_area,monthly_rent,employee_count,store_rating
0,ST00001,Readers Point Branch - Manikganj,Madaripur,Dhaka,Flagship,2025-12-19 23:07:19,Shaila Ahmed,2677.74,64043.41,20,4.92
1,ST00002,Rokomari Branch - Rajbari,Tangail,Dhaka,Mini,2017-08-01 00:53:13,Mahin Nahar,1206.76,86635.24,38,4.52
2,ST00003,BookHub Branch - Chittagong,Munshiganj,Dhaka,Standard,2021-07-19 12:44:17,Tanjim Bhuiyan,3820.77,224329.55,9,3.57
3,ST00004,Rokomari Branch - Brahmanbaria,Dhaka,Dhaka,Standard,2017-12-11 05:09:49,Raisa Uddin,3326.62,110611.43,47,3.39
4,ST00005,Novel Nest Center - Dhaka,Gazipur,Dhaka,Mini,2021-06-08 10:49:38,Sifat Amin,5898.30,85921.55,4,4.10


## Generate books / products

In [8]:
N_BOOKS = 1000
product_ids = [f"P{str(i).zfill(5)}" for i in range(1, N_BOOKS + 1)]

book_rows = []

for i in range(N_BOOKS):
    category = rng.choice(list(book_categories.keys()), p=[0.16, 0.10, 0.18, 0.11, 0.12, 0.10, 0.12, 0.07, 0.04])
    sub_category = rng.choice(book_categories[category])
    style = rng.choice(title_styles)
    audience = rng.choice(["Kids", "Young", "Professional", "Elderly"], p=[0.12, 0.50, 0.28, 0.10])

    topic = sub_category
    book_title = f"{topic} {style}"
    if category in ["Academic", "Exam Prep", "Programming"]:
        book_title += f" {rng.choice(['Edition', 'Series', 'Practice', 'Pro'])}"

    # realistic prices
    if category == "Children":
        unit_price = rng.uniform(120, 450)
    elif category in ["Academic", "Exam Prep", "Programming"]:
        unit_price = rng.uniform(250, 1200)
    elif category == "Business":
        unit_price = rng.uniform(300, 1500)
    elif category == "Religious":
        unit_price = rng.uniform(180, 900)
    else:
        unit_price = rng.uniform(200, 1000)

    unit_price = round(unit_price, 2)
    cost_price = round(unit_price * rng.uniform(0.55, 0.82), 2)

    # supplier selection aligned to category
    candidate_suppliers = supplier_df.loc[supplier_df["product_category"] == category, "supplier_id"]
    if len(candidate_suppliers) == 0:
        candidate_suppliers = supplier_df["supplier_id"]

    book_rows.append({
        "product_id": product_ids[i],
        "product_name": book_title,
        "category": category,
        "sub_category": sub_category,
        "author_name": f"{rng.choice(author_first)} {rng.choice(author_last)}",
        "publisher_name": supplier_df.loc[supplier_df["supplier_id"] == rng.choice(candidate_suppliers.to_numpy()), "supplier_name"].iloc[0],
        "unit_price": unit_price,
        "cost_price": cost_price,
        "supplier_id": rng.choice(candidate_suppliers.to_numpy()),
        "stock_quantity": int(rng.integers(15, 650)),
        "launch_date": random_dates("2019-01-01", "2026-06-12", 1)[0],
        "format": rng.choice(book_formats),
        "language": rng.choice(["Bangla", "English"], p=[0.70, 0.30]),
        "page_count": int(rng.integers(30, 850)),
        "reader_group": audience,
        "best_seller_score": round(float(rng.uniform(0.2, 1.0)), 3)
    })

book_df = pd.DataFrame(book_rows)
book_df.head()

,product_id,product_name,category,sub_category,author_name,publisher_name,unit_price,cost_price,supplier_id,stock_quantity,launch_date,format,language,page_count,reader_group,best_seller_score
0,P00001,Short Stories Masterclass,Fiction,Short Stories,Amin Ahmed,Green Books Ltd,454.53,312.40,S00483,453,2022-10-18 21:54:10,Hardcover,Bangla,277,Professional,0.984
1,P00002,Entrepreneurship Handbook,Business,Entrepreneurship,Naila Uddin,Prime Supply Co,518.21,334.91,S00143,244,2021-04-04 15:28:24,Paperback,Bangla,841,Kids,0.455
2,P00003,Databases Handbook Series,Programming,Databases,Jalal Islam,Bright Supply Co,1001.02,625.99,S00970,553,2020-09-06 21:29:41,eBook,English,842,Elderly,0.314
3,P00004,English Grammar Masterclass Series,Academic,English Grammar,Mala Chowdhury,Eastern Trade Group,537.49,403.33,S00982,238,2021-12-25 03:47:51,eBook,English,674,Young,0.991
4,P00005,Bank Jobs Guide Practice,Exam Prep,Bank Jobs,Mohiuddin Rahman,Delta Supply Co,930.27,715.75,S00214,482,2021-08-04 05:55:44,eBook,English,675,Kids,0.503


## Generate customers

In [9]:
N_CUSTOMERS = 1000
customer_ids = [f"C{str(i).zfill(5)}" for i in range(1, N_CUSTOMERS + 1)]

# Young customers must dominate the dataset
reader_group = rng.choice(
    ["Kids", "Young", "Professional", "Elderly"],
    size=N_CUSTOMERS,
    p=[0.10, 0.50, 0.30, 0.10]
)

age = []
for g in reader_group:
    if g == "Kids":
        age.append(int(rng.integers(6, 13)))
    elif g == "Young":
        age.append(int(rng.integers(18, 30)))
    elif g == "Professional":
        age.append(int(rng.integers(30, 56)))
    else:
        age.append(int(rng.integers(56, 76)))

age = np.array(age)

loyalty_score = np.clip(np.round(rng.normal(62, 18, size=N_CUSTOMERS), 0), 1, 100)
customer_segment = pd.cut(
    loyalty_score,
    bins=[0, 40, 60, 80, 100],
    labels=["Bronze", "Silver", "Gold", "Platinum"],
    include_lowest=True
).astype(str)

customer_city_weights = np.array([0.25 if d == "Dhaka" else 0.05 for d in districts], dtype=float)
customer_city_weights = customer_city_weights / customer_city_weights.sum()

customer_df = pd.DataFrame({
    "customer_id": customer_ids,
    "customer_name": make_name(N_CUSTOMERS),
    "gender": rng.choice(genders, size=N_CUSTOMERS, p=[0.55, 0.45]),
    "age": age,
    "city": rng.choice(districts, size=N_CUSTOMERS, p=customer_city_weights),
    "region": None,
    "signup_date": random_dates("2021-01-01", "2026-06-12", N_CUSTOMERS),
    "customer_segment": customer_segment,
    "email": [make_email(name, domain=rng.choice(["gmail.com", "yahoo.com", "outlook.com"])) for name in make_name(N_CUSTOMERS)],
    "phone": make_phone(N_CUSTOMERS),
    "loyalty_score": loyalty_score,
    "reader_group": reader_group
})

customer_df["region"] = customer_df["city"].map(district_to_region)
customer_df.head()

,customer_id,customer_name,gender,age,city,region,signup_date,customer_segment,email,phone,loyalty_score,reader_group
0,C00001,Sifat Sarker,Male,21,Brahmanbaria,Chattogram,2022-10-08 18:26:30,Bronze,nafis.khan@gmail.com,01573681081,34.0,Young
1,C00002,Sami Sheikh,Male,24,Dhaka,Dhaka,2024-10-25 10:49:31,Gold,sami.uddin@yahoo.com,01391624938,63.0,Young
2,C00003,Nafis Khan,Male,21,Gazipur,Dhaka,2023-12-20 07:28:35,Gold,nusrat.das@outlook.com,01959490272,77.0,Young
3,C00004,Shuvo Uddin,Male,34,Rangamati,Chattogram,2021-07-17 06:43:21,Gold,rafi.sheikh@outlook.com,01423607802,80.0,Professional
4,C00005,Nusrat Miah,Female,19,Munshiganj,Dhaka,2022-03-01 20:23:54,Silver,hasan.miah@outlook.com,01781413381,55.0,Young


## Generate sales

In [10]:
N_SALES = 2000
sales_ids = [f"SLS{str(i).zfill(6)}" for i in range(1, N_SALES + 1)]

# Sampling weights: young customers purchase more frequently
customer_weight_map = {"Kids": 1.2, "Young": 3.8, "Professional": 2.4, "Elderly": 1.0}
customer_weights = customer_df["reader_group"].map(customer_weight_map).to_numpy()
customer_weights = customer_weights / customer_weights.sum()

book_weights = book_df["best_seller_score"].to_numpy()
book_weights = book_weights / book_weights.sum()

store_weights = np.array([2.5 if s == "Dhaka" else 1.0 for s in store_df["city"]], dtype=float)
store_weights = store_weights / store_weights.sum()

sales_customer_ids = rng.choice(customer_df["customer_id"], size=N_SALES, p=customer_weights)
sales_product_ids = rng.choice(book_df["product_id"], size=N_SALES, p=book_weights)
sales_store_ids = rng.choice(store_df["store_id"], size=N_SALES, p=store_weights)
sales_dates = random_dates("2024-01-01", "2026-06-12", N_SALES)

sales_df = pd.DataFrame({
    "sales_id": sales_ids,
    "customer_id": sales_customer_ids,
    "product_id": sales_product_ids,
    "store_id": sales_store_ids,
    "sales_date": sales_dates
})

# Join only for generating realistic sale behavior
tmp = (
    sales_df
    .merge(customer_df[["customer_id", "age", "reader_group", "city", "region"]], on="customer_id", how="left")
    .merge(book_df[["product_id", "category", "sub_category", "unit_price", "cost_price", "supplier_id"]], on="product_id", how="left")
    .merge(store_df[["store_id", "city", "region"]].rename(columns={"city": "store_city", "region": "store_region"}), on="store_id", how="left")
)

# Quantity is influenced by age group and category
quantity = []
for _, row in tmp.iterrows():
    if row["reader_group"] == "Kids":
        q = int(rng.integers(1, 4))
    elif row["reader_group"] == "Young":
        q = int(rng.integers(1, 5))
    elif row["reader_group"] == "Professional":
        q = int(rng.integers(1, 3))
    else:
        q = int(rng.integers(1, 3))

    # some categories are usually bought in small quantity
    if row["category"] in ["Academic", "Programming", "Business", "Exam Prep"]:
        q = min(q, 3)

    quantity.append(q)

tmp["quantity"] = quantity

# Discounts vary by channel
tmp["sales_channel"] = rng.choice(
    ["Online", "In-Store", "App"],
    size=N_SALES,
    p=[0.45, 0.35, 0.20]
)

tmp["payment_method"] = rng.choice(payment_methods, size=N_SALES, p=[0.18, 0.22, 0.25, 0.23, 0.12])

# more realistic discounts
base_discount = np.where(tmp["sales_channel"].eq("Online"), rng.normal(0.10, 0.05, size=N_SALES), rng.normal(0.05, 0.03, size=N_SALES))
tmp["discount"] = np.clip(np.round(base_discount, 2), 0.00, 0.30)

tmp["total_amount"] = np.round(tmp["quantity"] * tmp["unit_price"] * (1 - tmp["discount"]), 2)

sales_df = tmp[[
    "sales_id","supplier_id", "customer_id", "product_id", "store_id", "sales_date",
    "quantity", "unit_price", "discount", "total_amount",
    "payment_method", "sales_channel", "reader_group", "category"
]].copy()

sales_df.head()

,sales_id,supplier_id,customer_id,product_id,store_id,sales_date,quantity,unit_price,discount,total_amount,payment_method,sales_channel,reader_group,category
0,SLS000001,S00556,C00980,P00204,ST00388,2025-05-30 15:08:08,3,297.77,0.14,768.25,Nagad,Online,Young,Biography
1,SLS000002,S00179,C00104,P00109,ST00166,2025-05-05 10:17:59,1,397.12,0.11,353.44,Card,Online,Young,Programming
2,SLS000003,S00608,C00073,P00238,ST00634,2024-08-27 09:24:18,1,659.86,0.08,607.07,Nagad,Online,Professional,Fiction
3,SLS000004,S00989,C00530,P00503,ST00285,2025-02-10 12:02:48,3,399.02,0.12,1053.41,Nagad,Online,Young,Business
4,SLS000005,S00794,C00612,P00243,ST00634,2025-05-14 11:04:42,3,1098.68,0.16,2768.67,Card,Online,Young,Academic


In [11]:
import pandas as pd

path = r"F:/UIU/2nd sems course/Data Analytics/Power BI Project/Data Visualization Assignment/Python Files/Excel Synthetic Data/sales_raw_messy.csv"

df = pd.read_csv(path)

# convert to datetime (invalid → NaT)
df["sales_date"] = pd.to_datetime(df["sales_date"], errors="coerce")

# replace invalid dates with fallback OR keep as NaT
df["sales_date"] = df["sales_date"].fillna(pd.Timestamp("2024-01-01"))

# ensure proper format for PostgreSQL
df["sales_date"] = df["sales_date"].dt.date

# overwrite file
df.to_csv(path, index=False)

print("sales_date cleaned and file updated")

sales_date cleaned and file updated


## Deliberately make the raw data messy

In [12]:
# Keep a raw copy before corruption
customers_raw = customer_df.copy()
books_raw = book_df.copy()
suppliers_raw = supplier_df.copy()
stores_raw = store_df.copy()
sales_raw = sales_df.copy()

# -----------------------------
# 1) Missing / null values
# -----------------------------
customers_raw = introduce_nulls(customers_raw, ["age", "city", "email", "phone", "loyalty_score"], frac=0.04)
books_raw = introduce_nulls(books_raw, ["unit_price", "cost_price", "stock_quantity", "language", "format"], frac=0.03)
suppliers_raw = introduce_nulls(suppliers_raw, ["country", "phone", "email", "rating"], frac=0.03)
stores_raw = introduce_nulls(stores_raw, ["monthly_rent", "employee_count", "store_rating"], frac=0.03)
sales_raw = introduce_nulls(sales_raw, ["quantity", "discount", "sales_channel", "payment_method"], frac=0.04)

# -----------------------------
# 2) Inconsistent text formats
# -----------------------------
customers_raw = introduce_inconsistent_text(customers_raw, "city", [
    lambda x: x.lower(),
    lambda x: x.upper(),
    lambda x: f" {x} ",
    lambda x: x.title()
])

books_raw = introduce_inconsistent_text(books_raw, "category", [
    lambda x: x.lower(),
    lambda x: x.upper(),
    lambda x: f" {x} "
])

sales_raw = introduce_inconsistent_text(sales_raw, "sales_channel", [
    lambda x: x.lower(),
    lambda x: x.upper(),
    lambda x: f" {x} "
])

# -----------------------------
# 3) Invalid values / outliers
# -----------------------------
# customer age outliers and invalid ages
bad_age_idx = rng.choice(customers_raw.index, size=8, replace=False)
customers_raw.loc[bad_age_idx[:3], "age"] = [0, 150, -5]
customers_raw.loc[bad_age_idx[3:], "age"] = [98, 120, 2, 200, -10]

# unrealistic loyalty scores
customers_raw.loc[rng.choice(customers_raw.index, size=5, replace=False), "loyalty_score"] = [250, -20, 180, 300, -5]

# negative and extreme prices
bad_book_idx = rng.choice(books_raw.index, size=8, replace=False)
books_raw.loc[bad_book_idx[:4], "unit_price"] = [5000, -100, 20000, 0]
books_raw.loc[bad_book_idx[4:], "cost_price"] = [-50, 9000, 100000, 0]

# invalid sales quantities / discounts
bad_sales_idx = rng.choice(sales_raw.index, size=10, replace=False)
sales_raw.loc[bad_sales_idx[:4], "quantity"] = [0, -2, 40, 100]
sales_raw.loc[bad_sales_idx[4:8], "discount"] = [1.5, -0.2, 2.0, 0.95]
sales_raw.loc[bad_sales_idx[8:], "sales_date"] = ["2026-15-40", "not a date"]

# inconsistent district names
city_variants = {
    "dhaka": " DHAKA ",
    "chittagong": "chittagong",
    "comilla": " Cumilla ",
    "cox's bazar": "COX'S BAZAR"
}
for df in [customers_raw, stores_raw, suppliers_raw]:
    if "city" in df.columns:
        for canonical, variant in city_variants.items():
            mask = df["city"].astype(str).str.strip().str.lower() == canonical
            df.loc[mask, "city"] = variant

# some invalid emails/phones
customers_raw.loc[rng.choice(customers_raw.index, size=5, replace=False), "email"] = ["bad_email", "abc@", "user@@mail.com", "not-an-email", np.nan]
customers_raw.loc[rng.choice(customers_raw.index, size=5, replace=False), "phone"] = ["123", "000", "ABCD1234567", "017", np.nan]

# some sales dates as strings to simulate dirty import
sales_raw["sales_date"] = sales_raw["sales_date"].astype("object")

print("Raw tables created with deliberate quality issues.")
# ==========================================
# SAVE RAW (MESSY) DATASETS
# ==========================================

customers_raw.to_csv(
    OUT_DIR / "customers_raw_messy.csv",
    index=False
)

books_raw.to_csv(
    OUT_DIR / "books_raw_messy.csv",
    index=False
)

suppliers_raw.to_csv(
    OUT_DIR / "suppliers_raw_messy.csv",
    index=False
)

stores_raw.to_csv(
    OUT_DIR / "stores_raw_messy.csv",
    index=False
)

sales_raw.to_csv(
    OUT_DIR / "sales_raw_messy.csv",
    index=False
)

print("\nAll messy datasets saved successfully.")
print(OUT_DIR)

Raw tables created with deliberate quality issues.

All messy datasets saved successfully.
F:\UIU\2nd sems course\Data Analytics\Power BI Project\Data Visualization Assignment\Python Files\test


C:\Users\Administrator\AppData\Local\Temp\ipykernel_24844\927614970.py:59: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['2026-15-40', 'not a date']' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  sales_raw.loc[bad_sales_idx[8:], "sales_date"] = ["2026-15-40", "not a date"]
